In [1]:
#Install/Import libraries

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
#Load the Telco Churn dataset
# Using a public copy of the IBM Telco Customer Churn dataset

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
#Inspect and clean the data
# Check data types and missing values

print(df.info())
print(df.isnull().sum())

# TotalCharges sometimes has blank strings instead of NaN — fix that
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop customerID — it's just an identifier, not useful for prediction
df.drop("customerID", axis=1, inplace=True)

# Target column: Churn (Yes/No) -> convert to binary (1/0)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Missing values after cleaning:\n", df.isnull().sum().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
#Split features and target

X = df.drop("Churn", axis=1)
y = df["Churn"]

# Identify numeric and categorical columns automatically
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", numeric_features)
print("Categorical columns:", categorical_features)

Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [5]:
#Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)

Train size: (5634, 19) Test size: (1409, 19)


In [6]:
#Build preprocessing pipeline

# Numeric pipeline: fill missing values, then scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Categorical pipeline: fill missing values, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# Combine both into a single ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [7]:
#Build full pipelines (preprocessing + model)

# Pipeline 1: Logistic Regression
log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

# Pipeline 2: Random Forest
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)),
])

In [8]:
#Hyperparameter tuning with GridSearchCV (Logistic Regression)

log_reg_params = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__penalty": ["l2"],
}

log_reg_grid = GridSearchCV(
    log_reg_pipeline,
    param_grid=log_reg_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

log_reg_grid.fit(X_train, y_train)

print("Best Logistic Regression params:", log_reg_grid.best_params_)
print("Best CV accuracy:", log_reg_grid.best_score_)

Best Logistic Regression params: {'classifier__C': 10, 'classifier__penalty': 'l2'}
Best CV accuracy: 0.8045768249380222


In [9]:
#Hyperparameter tuning with GridSearchCV (Random Forest)

rf_params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5],
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=rf_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest params:", rf_grid.best_params_)
print("Best CV accuracy:", rf_grid.best_score_)

Best Random Forest params: {'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Best CV accuracy: 0.7997845551070841


In [10]:
#Evaluate both models on test set

# Evaluate Logistic Regression
log_reg_preds = log_reg_grid.predict(X_test)
print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, log_reg_preds))
print(classification_report(y_test, log_reg_preds))

# Evaluate Random Forest
rf_preds = rf_grid.predict(X_test)
print("=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, rf_preds))
print(classification_report(y_test, rf_preds))

=== Logistic Regression ===
Accuracy: 0.8055358410220014
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

=== Random Forest ===
Accuracy: 0.7998580553584103
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [11]:
#Pick the best model and export it with joblib

# Compare test accuracy and choose the better pipeline
log_reg_acc = accuracy_score(y_test, log_reg_preds)
rf_acc = accuracy_score(y_test, rf_preds)

if rf_acc >= log_reg_acc:
    best_pipeline = rf_grid.best_estimator_
    best_name = "Random Forest"
else:
    best_pipeline = log_reg_grid.best_estimator_
    best_name = "Logistic Regression"

print(f"Best model: {best_name} (Test Accuracy: {max(log_reg_acc, rf_acc):.4f})")

# Save the entire pipeline (preprocessing + trained model) as one file
joblib.dump(best_pipeline, "churn_prediction_pipeline.pkl")
print("Pipeline saved as churn_prediction_pipeline.pkl")

Best model: Logistic Regression (Test Accuracy: 0.8055)
Pipeline saved as churn_prediction_pipeline.pkl


In [12]:
#Test loading the saved pipeline (sanity check)

# Reload the saved pipeline and confirm it works on new data
loaded_pipeline = joblib.load("churn_prediction_pipeline.pkl")

sample_preds = loaded_pipeline.predict(X_test.iloc[:5])
print("Sample predictions:", sample_preds)
print("Actual values:      ", y_test.iloc[:5].values)

Sample predictions: [0 1 0 0 0]
Actual values:       [0 0 0 0 0]
